# Redis cache — service test

Tests `CacheClient`: key generation, store, hit, and key-sensitivity.

**Prereqs:** `docker compose up -d redis`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from src.config import get_settings
from src.services.cache.factory import make_cache_client

cache = make_cache_client(get_settings())   # pings Redis — fails loud if down
print('connected, TTL seconds:', cache.ttl_seconds)

In [ ]:
from src.schemas.api.ask import AskRequest, AskResponse

req = AskRequest(query='What is attention?', top_k=3, use_hybrid=True)
resp = AskResponse(query=req.query, answer='Attention is...', sources=[], chunks_used=3, search_mode='hybrid')

# Key is a hash of normalized query + params
print('cache key:', cache._generate_cache_key(req))

In [ ]:
# Miss → store → hit
print('before store:', await cache.find_cached_response(req))
await cache.store_response(req, resp)
hit = await cache.find_cached_response(req)
print('after store :', hit.answer if hit else None)

In [ ]:
# Key sensitivity: different top_k → different key → MISS (correctness!)
req_k5 = AskRequest(query='What is attention?', top_k=5, use_hybrid=True)
print('same query, top_k=5:', await cache.find_cached_response(req_k5), '(expect None)')

# But whitespace/case-normalized query → same key → HIT
req_ws = AskRequest(query='  WHAT IS ATTENTION?  ', top_k=3, use_hybrid=True)
hit = await cache.find_cached_response(req_ws)
print('normalized query    :', hit.answer if hit else None, '(expect the answer)')

In [ ]:
# Peek at raw Redis + cleanup
keys = cache.redis.keys('exact_cache:*')
print('keys in redis:', keys)
for k in keys:
    cache.redis.delete(k)
print('cleaned up')